## Perform trajectory analysis on VASA data

## Load libraries

In [ ]:
import os
import sys
import random
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

import random

random.seed(10)

from datetime import datetime    
from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import scFates as scf
import logging
import cellrank as cr
import scvelo as scv
import matplotlib.pyplot as plt
import seaborn as sns
import marsilea as ma
import marsilea.plotter as mp
from matplotlib.patches import Rectangle
from pathlib import Path

# Fix pandas .iteritems() if using pandas >= 2
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items

sns.reset_orig()
%matplotlib inline

logging.getLogger('matplotlib.font_manager').disabled = True

sc.set_figure_params(figsize=(10, 10))
scf.set_figure_pubready()

figdir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/VASA")
figdir.mkdir(parents=True, exist_ok=True)

tablesdir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tables/VASA")
tablesdir.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
# Define palette for the lineages
palette = {
    "Late ECs": "#ee8865",
    "Late X cells": "#85a7cf",
    "D cells": "#6ac077",
    "K cells": "#58B6D7"
}

In [ ]:
# Load the dataset
adata_orig = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.h5ad")

# Remove SMOC2+ cells
adata_orig = adata_orig[adata_orig.obs["cell_type_initial"] != "SMOC2+ cells"]

# Load data
adata = sc.read("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.h5ad")
adata = adata.raw.to_adata()

# Add approriate colors for the terminal states
adata.uns['term_states_fwd_colors'] = ["#ee8865","#85a7cf", "#6ac077", "#58B6D7"]

# Standard preprocessing
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

## Load TFs

In [ ]:
# Get list of human TFs
human_tfs = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/external/TF_names_v_1.01.txt", header=None)[0].tolist()
# Add Percc1 
human_tfs.append("PERCC1")#print(f"Number of TFs in the dataset: {len(human_tfs)}")

In [ ]:
# Load perturbed conditions
with open('/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_conditions.txt', "r") as f:
    tf_perturbations = [line.strip() for line in f.readlines()]
    # Make upper case to match the gene names in the dataset
    tf_perturbations = [tf.upper() for tf in tf_perturbations]

# Some TFs have a different name i.e. POU2AF2 -> POU2F2, so we need to rename them
# Create a mapping of the TFs that need to be renamed
tf_rename_mapping = {
    "POU2AF2": "POU2F2",
    "POU2AF3": "POU2F3",
    "HHMGB3": "HMGB3",
    "HETS1": "ETS1",
    "HETV4": "ETV4"
}
# Rename the TFs in the perturbation list
tf_perturbations = [tf_rename_mapping.get(tf, tf) for tf in tf_perturbations]

In [ ]:
# Create the union of tf_perturbations and human_tfs and subset to those that are in the dataset
tf_union = set(tf_perturbations) | set(human_tfs)
tf_union_in_dataset = [tf for tf in tf_union if tf in adata.var_names]
print(f"Number of TFs in the union of perturbations and human TFs that are in the dataset: {len(tf_union_in_dataset)}")

# Subset the tf_union_in_dataset to those those that are in tf_perturbations 
human_tfs_in_perturbations = [tf for tf in tf_union_in_dataset if tf in tf_perturbations]
print(f"Number of TFs in the union of perturbations and human TFs that are in the dataset and in the perturbations: {len(human_tfs_in_perturbations)}")

## Load drivers

In [ ]:
# Load the fate drivers table
fate_drivers_df = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.v3.annotated.cellrank.lineage.drivers.csv", index_col=0)

# Select only _corr
fate_drivers_df_corr = fate_drivers_df.filter(like="_corr")

# Pivot longer to have one row per gene and lineage
fate_drivers_long = fate_drivers_df_corr.reset_index().melt(id_vars="index", var_name="lineage", value_name="correlation").rename(columns={"index": "gene"})
fate_drivers_long["lineage"] = fate_drivers_long["lineage"].str.replace("_corr", "")
fate_drivers_long = fate_drivers_long[["gene", "lineage", "correlation"]]

# Do the same with qval
fate_drivers_qval = fate_drivers_df.filter(like="_qval")
fate_drivers_qval_long = fate_drivers_qval.reset_index().melt(id_vars="index", var_name="lineage", value_name="qval").rename(columns={"index": "gene"})
fate_drivers_qval_long["lineage"] = fate_drivers_qval_long["lineage"].str.replace("_qval", "")
fate_drivers_qval_long = fate_drivers_qval_long[["gene", "lineage", "qval"]]

# Make a combined dataframe with both correlation and qval
fate_drivers_combined = pd.merge(fate_drivers_long, fate_drivers_qval_long, on=["gene", "lineage"])

In [ ]:
fate_drivers_df

## Plot all significant drivers

In [ ]:
# Subset to significant drivers (qval < 0.05) with a correlation > 0.1
significant_drivers = fate_drivers_combined[(fate_drivers_combined["qval"] < 0.05) & (fate_drivers_combined["correlation"] > 0.1)]

# Subset to intersection 
significant_drivers = significant_drivers[significant_drivers["gene"].isin(tf_union_in_dataset)]["gene"].unique().tolist()

# Subset original fate_drivers_combined to only significant drivers
fate_drivers_tf_df = fate_drivers_combined[fate_drivers_combined["gene"].isin(significant_drivers)]
fate_drivers_tf_df = fate_drivers_tf_df.pivot_table(index="gene", columns="lineage", values="correlation")

# Sort the order of the lineages [Late ECs, Late X cells, D cells, K cells]
fate_drivers_tf_df = fate_drivers_tf_df[["Late ECs", "Late X cells", "D cells", "K cells"]]

# Sort the order of the TFs based on the lineage they are most correlated with
max_group = fate_drivers_tf_df.idxmax(axis=1)
ordered = []
groups = []
for group in fate_drivers_tf_df.columns:
    # Sort the TFs within each group by their correlation value with that group
    group_tfs = max_group[max_group == group].index.tolist()
    group_tfs_sorted = fate_drivers_tf_df.loc[group_tfs, group].sort_values(ascending=False).index.tolist()
    ordered.extend(group_tfs_sorted)
    groups.extend([group] * len(group_tfs_sorted))
fate_drivers_tf_df = fate_drivers_tf_df.loc[ordered]

# Clip negative values to zero because not relevant
fate_drivers_tf_df = fate_drivers_tf_df.clip(lower=0)

# Get cut points where the groups label changes
group_change_points = np.where(np.array(groups[:-1]) != np.array(groups[1:]))[0] + 1

In [ ]:
fate_drivers_combined

In [ ]:
fate_drivers_tf_df.to_csv(tablesdir / "fate_drivers_tf_df_v3.csv")

In [ ]:
col_colors = [palette[g] for g in groups]

with plt.rc_context({"axes.grid": False}):
    g = sns.clustermap(
        fate_drivers_tf_df.T,
        cmap="Greys",
        figsize=(10, 4),
        row_cluster=False,
        col_cluster=False,
        col_colors=col_colors,
        vmin=0,
        vmax=0.5,
        cbar_kws={"label": "Correlation with\nfate probability"}
    )

# Label only exemplary TFs on x-axis
label_tfs = ["LMX1A", "LMX1B", "PERCC1", "ARX", "HHEX", "ISL1", "PAX6", "GATA4",
             "PAX4", "RFX6", "RFX3", "PROX1", "ZCCHC12"]

g.ax_heatmap.set_xticks([])

for tf in label_tfs:
    if tf in fate_drivers_tf_df.index:
        idx = fate_drivers_tf_df.index.get_loc(tf)
        g.ax_heatmap.text(
            idx + 4, -0.3, tf,
            va="bottom",
            ha="right",
            rotation=90,
            fontsize=8
        )

pos = g.ax_col_colors.get_position()
g.ax_col_colors.set_position([
    pos.x0,
    pos.y0,
    pos.width,
    pos.height * 3
])

# Fix row labels
g.ax_heatmap.set_yticks(range(len(fate_drivers_tf_df.columns)))
g.ax_heatmap.set_yticklabels(fate_drivers_tf_df.columns, rotation=0)

# Add lines to separate the groups
for change_point in group_change_points:
    g.ax_heatmap.axvline(change_point, color="black", linestyle="--", linewidth=1)

# Add a horizontal line for each group
for i in range(len(fate_drivers_tf_df.columns)):
    g.ax_heatmap.axhline(i, color="black", linestyle="--", linewidth=1)


# Add border
for spine in g.ax_heatmap.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)

g.savefig(figdir / "fate_drivers_clustermap_v3.pdf", bbox_inches="tight")

### Plot all perturbed drivers

In [ ]:
# Subset to significant drivers (qval < 0.05) with a correlation > 0.1
significant_drivers = fate_drivers_combined[(fate_drivers_combined["qval"] < 0.05) & (fate_drivers_combined["correlation"] > 0.1)]

# Subset to intersection 
significant_drivers = significant_drivers[significant_drivers["gene"].isin(human_tfs_in_perturbations)]["gene"].unique().tolist()

# Subset original fate_drivers_combined to only significant drivers
fate_drivers_tf_df = fate_drivers_combined[fate_drivers_combined["gene"].isin(significant_drivers)]
fate_drivers_tf_df = fate_drivers_tf_df.pivot_table(index="gene", columns="lineage", values="correlation")

# Sort the order of the lineages [Late ECs, Late X cells, D cells, K cells]
fate_drivers_tf_df = fate_drivers_tf_df[["Late ECs", "Late X cells", "D cells", "K cells"]]

# Sort the order of the TFs based on the lineage they are most correlated with
max_group = fate_drivers_tf_df.idxmax(axis=1)
ordered = []
groups = []
for group in fate_drivers_tf_df.columns:
    # Sort the TFs within each group by their correlation value with that group
    group_tfs = max_group[max_group == group].index.tolist()
    group_tfs_sorted = fate_drivers_tf_df.loc[group_tfs, group].sort_values(ascending=False).index.tolist()
    ordered.extend(group_tfs_sorted)
    groups.extend([group] * len(group_tfs_sorted))
fate_drivers_tf_df = fate_drivers_tf_df.loc[ordered]

# Clip negative values to zero because not relevant
fate_drivers_tf_df = fate_drivers_tf_df.clip(lower=0)

# Get cut points where the groups label changes
group_change_points = np.where(np.array(groups[:-1]) != np.array(groups[1:]))[0] + 1

In [ ]:
fate_drivers_tf_df

In [ ]:
# Get all TFs in human_tfs_in_perturbations that are not in fate_drivers_tf_df.index
missing_tfs = [tf for tf in human_tfs_in_perturbations if tf not in fate_drivers_tf_df.index]
print(f"TFs in perturbations that are not significant drivers: {missing_tfs}")


In [ ]:
col_colors = [palette[g] for g in groups]

with plt.rc_context({"axes.grid": False}):
    g = sns.clustermap(
        fate_drivers_tf_df.T,
        cmap="Greys",
        figsize=(30, 4),
        row_cluster=False,
        col_cluster=False,
        col_colors=col_colors,
        cbar_kws={"label": "Correlation with\nfate probability"}
    )

pos = g.ax_col_colors.get_position()
g.ax_col_colors.set_position([
    pos.x0,
    pos.y0,
    pos.width,
    pos.height * 3
])

# Fix row labels
g.ax_heatmap.set_yticks(range(len(fate_drivers_tf_df.columns)))
g.ax_heatmap.set_yticklabels(fate_drivers_tf_df.columns, rotation=0)

# Add border
for spine in g.ax_heatmap.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.2)

g.savefig(figdir / "perturbed_drivers_clustermap_v3.pdf", bbox_inches="tight")